# Exploration des données brutes Olist

**Objectif :** comprendre les données **avant** toute transformation, pour justifier les choix du pipeline dbt.

Chaque section part d'une **observation** dans les CSV bruts et se termine par le **choix** qu'elle justifie.
Les choix sont résumés dans `docs/choix_de_modelisation.md`.

> On lit directement les fichiers de `ctc_olist/data/raw/`, sans passer par dbt.
> Pour les obtenir : `./scripts/download_data.sh`

## Préparation

DuckDB peut lire un CSV comme une table. La petite fonction `sql()` lance une requête et affiche le résultat en tableau.

In [ ]:
import duckdb

# Dossier des CSV bruts (le notebook est dans notebooks/)
RAW = "../ctc_olist/data/raw"

def sql(query):
    """Lance une requête SQL et renvoie un tableau pandas."""
    return duckdb.sql(query).df()

def csv(name):
    """Chemin d'un fichier brut, à utiliser dans une requête."""
    return f"'{RAW}/{name}.csv'"

## 1. Vue d'ensemble des fichiers

**Question :** combien de fichiers, de lignes et de colonnes ? Quels types DuckDB devine-t-il ?

In [ ]:
files = [
    "olist_orders_dataset",
    "olist_customers_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation",
    "olist_geolocation_dataset",
]

# Nombre de lignes et de colonnes de chaque fichier
import pandas as pd

rows = []
for f in files:
    n_rows = sql(f"select count(*) as n from {csv(f)}")["n"][0]
    n_cols = len(sql(f"select * from {csv(f)} limit 0").columns)
    rows.append({"file": f, "rows": n_rows, "columns": n_cols})

pd.DataFrame(rows)

**À retenir :** 9 fichiers. La géolocalisation est le plus gros, mais elle ne sert pas à notre sujet (la satisfaction client) : on ne la déclare pas comme source.

### Les types devinés par DuckDB

Exemple avec les commandes et les clients.

In [ ]:
sql(f"describe select * from {csv('olist_orders_dataset')}")

In [ ]:
sql(f"describe select * from {csv('olist_customers_dataset')}")

In [ ]:
# Des codes postaux commencent par 0 : en nombre, ce zéro disparaîtrait
sql(f"""
    select customer_zip_code_prefix
    from {csv('olist_customers_dataset')}
    where customer_zip_code_prefix like '0%'
    limit 5
""")

**À retenir :**
- Les types sont **devinés** à partir du contenu du fichier : ils pourraient changer si le fichier changeait.
- Certains noms ne correspondent pas au contenu : `order_delivered_carrier_date` contient une **heure**,
  et `order_estimated_delivery_date` est lue comme date et heure alors qu'elle est toujours à minuit.
- Les codes postaux commencent parfois par **0** : il faut les garder en **texte**.

**→ Choix :** en staging, on **convertit explicitement** chaque type (`cast`), on renomme avec la convention `_at` (date et heure) / `_date` (jour), et on garde les codes postaux en texte.

## 2. Clés et granularité

**Question :** dans chaque fichier, quelle colonne identifie une ligne de façon unique ?

Méthode : si le nombre de valeurs **distinctes** d'une colonne est égal au nombre de **lignes**, la colonne est unique.

In [ ]:
# Pour chaque candidat de clé : nombre de lignes et nombre de valeurs distinctes
checks = [
    ("olist_orders_dataset",              "order_id"),
    ("olist_customers_dataset",           "customer_id"),
    ("olist_customers_dataset",           "customer_unique_id"),
    ("olist_order_items_dataset",         "order_id"),
    ("olist_order_items_dataset",         "(order_id, order_item_id)"),
    ("olist_order_payments_dataset",      "(order_id, payment_sequential)"),
    ("olist_order_reviews_dataset",       "review_id"),
    ("olist_order_reviews_dataset",       "order_id"),
    ("olist_order_reviews_dataset",       "(review_id, order_id)"),
    ("olist_products_dataset",            "product_id"),
    ("olist_sellers_dataset",             "seller_id"),
    ("product_category_name_translation", "product_category_name"),
]

results = []
for file, key in checks:
    r = sql(f"select count(*) as n_rows, count(distinct {key}) as n_distinct from {csv(file)}")
    n_rows, n_distinct = r["n_rows"][0], r["n_distinct"][0]
    results.append({"file": file, "key": key, "rows": n_rows,
                    "distinct": n_distinct, "unique": n_rows == n_distinct})

pd.DataFrame(results)

**À retenir :**
- Commandes, produits, vendeurs : **une colonne suffit** (`order_id`, `product_id`, `seller_id`).
- Articles et paiements : il faut **deux colonnes** (clé composée).
- Avis : ni `review_id` ni `order_id` ne sont uniques seuls. Seule la paire l'est.
- Clients : `customer_id` est unique, mais **pas** `customer_unique_id`.

### Un client, plusieurs `customer_id`

**Question :** combien de `customer_id` différents pour une même personne ?

In [ ]:
# Nombre de customer_id par personne, puis nombre de personnes dans chaque cas
sql(f"""
    with per_person as (
        select customer_unique_id, count(*) as nb_customer_id
        from {csv('olist_customers_dataset')}
        group by customer_unique_id
    )
    select nb_customer_id, count(*) as nb_persons
    from per_person
    group by nb_customer_id
    order by nb_customer_id
""")

In [ ]:
# Chaque commande a son propre customer_id
sql(f"""
    select
        count(*)                    as orders,
        count(distinct customer_id) as distinct_customer_id
    from {csv('olist_orders_dataset')}
""")

**À retenir :** Olist crée un **nouveau `customer_id` à chaque commande** (99 441 commandes = 99 441 `customer_id`).
Une même personne qui commande plusieurs fois a donc plusieurs `customer_id`, mais un seul `customer_unique_id`.

**→ Choix :**
- Chaque modèle de staging garde la **granularité de sa source**, avec une clé composée quand il le faut (articles, paiements, avis).
- `dim_customers` a pour granularité **la personne** (`customer_unique_id`), sinon un client fidèle compterait pour plusieurs clients.

## 3. Les clients avec plusieurs adresses

**Question :** une même personne a-t-elle toujours la même adresse d'une commande à l'autre ?

Chaque `customer_id` a sa propre adresse. Comme une personne peut avoir plusieurs `customer_id` (section 2), elle peut aussi avoir plusieurs adresses.

In [ ]:
# Nombre d'adresses différentes (code postal + ville + état) par personne
sql(f"""
    with addresses as (
        select
            customer_unique_id,
            count(distinct customer_zip_code_prefix || customer_city || customer_state) as nb_addresses
        from {csv('olist_customers_dataset')}
        group by customer_unique_id
    )
    select
        count(*)                      as persons,
        cast(count_if(nb_addresses > 1) as integer) as persons_with_several_addresses
    from addresses
""")

### Un exemple : les adresses d'une personne dans le temps

In [ ]:
# Une personne avec plusieurs adresses, commande par commande
sql(f"""
    with movers as (
        select customer_unique_id
        from {csv('olist_customers_dataset')}
        group by customer_unique_id
        having count(distinct customer_city) > 1
        order by customer_unique_id
        limit 1
    )
    select
        c.customer_unique_id,
        o.order_purchase_timestamp,
        c.customer_city,
        c.customer_state
    from {csv('olist_customers_dataset')} c
    join {csv('olist_orders_dataset')} o on c.customer_id = o.customer_id
    where c.customer_unique_id in (select customer_unique_id from movers)
    order by o.order_purchase_timestamp
""")

**À retenir :**
- Seulement **252 personnes sur 96 096** (0,3 %) ont plusieurs adresses.
- Dans l'exemple, la personne commande en mai 2017 puis en juillet 2018, dans deux villes différentes.
- Ce n'est pas toujours un déménagement : l'adresse est celle de la **livraison** (un cadeau envoyé ailleurs, par exemple).
- Il faut quand même choisir **quelle adresse** garder dans la dimension client.

**→ Choix :** `dim_customers` utilise la technique d'historisation **SCD type 1** (dimension à évolution lente) : on garde seulement la **dernière adresse connue** (celle de la commande la plus récente) et on écrase les anciennes.

Le type 2 (garder l'historique, une ligne par adresse) serait plus complet, mais inutile ici : les déménagements sont trop rares pour changer l'analyse.

## 4. Les commandes avec plusieurs avis

**Question :** une commande a-t-elle toujours un seul avis ? Si non, les avis disent-ils la même chose ?

In [ ]:
# Nombre d'avis par commande, puis nombre de commandes dans chaque cas
sql(f"""
    with per_order as (
        select order_id, count(*) as nb_reviews
        from {csv('olist_order_reviews_dataset')}
        group by order_id
    )
    select nb_reviews, count(*) as nb_orders
    from per_order
    group by nb_reviews
    order by nb_reviews
""")

In [ ]:
# Parmi les commandes à plusieurs avis : combien ont des notes différentes ?
sql(f"""
    with multi as (
        select order_id, count(distinct review_score) as nb_scores
        from {csv('olist_order_reviews_dataset')}
        group by order_id
        having count(*) > 1
    )
    select
        count(*)                               as orders_with_several_reviews,
        cast(count_if(nb_scores > 1) as integer) as with_different_scores
    from multi
""")

### Un exemple

In [ ]:
# Les avis d'une commande qui en a plusieurs, du plus ancien au plus récent
sql(f"""
    with multi as (
        select order_id
        from {csv('olist_order_reviews_dataset')}
        group by order_id
        having count(*) > 1
        order by order_id
        limit 1
    )
    select order_id, review_id, review_score, review_creation_date, review_answer_timestamp
    from {csv('olist_order_reviews_dataset')}
    where order_id in (select order_id from multi)
    order by review_answer_timestamp
""")

**À retenir :**
- **547 commandes** ont 2 ou 3 avis (0,6 % des commandes).
- Pour **202** d'entre elles, les notes sont **différentes** : le choix de l'avis change le résultat.
- Dans l'exemple, le client a répondu deux fois, à quelques jours d'intervalle.

**→ Choix :** dans `int_review_sentiment`, on garde **l'avis le plus récent** de chaque commande (`review_answer_timestamp`).
C'est l'opinion la plus à jour du client, et on obtient **une ligne par commande**, la granularité de `fct_orders`.

## 5. Les valeurs manquantes

**Question :** où manque-t-il des valeurs, et pourquoi ? Trois cas importants pour le modèle.

### 5.1 Les dates de livraison

In [ ]:
# Commandes sans date de livraison au client, par statut
sql(f"""
    select
        order_status,
        count(*)                                                       as orders,
        cast(count_if(order_delivered_customer_date is null) as integer) as without_delivery_date
    from {csv('olist_orders_dataset')}
    group by order_status
    order by orders desc
""")

**À retenir :** **2 965 commandes** n'ont pas de date de livraison. Ce n'est pas une erreur : elles sont en cours, annulées ou indisponibles.

**→ Choix :** dans `fct_orders`, le délai de livraison et le retard restent vides (`null`) pour ces commandes.
Le fait est relié à `dim_date` par la date d'achat seulement, qui n'est jamais vide.

### 5.2 Les catégories de produits

In [ ]:
# Produits sans catégorie
sql(f"""
    select
        count(*)                                                 as products,
        cast(count_if(product_category_name is null) as integer) as without_category
    from {csv('olist_products_dataset')}
""")

**À retenir :** **610 produits** n'ont pas de catégorie.

**→ Choix :** les produits ne sont pas modélisés dans l'entrepôt (le sujet d'analyse est la commande et la satisfaction). Cette anomalie n'a donc pas d'effet sur le pipeline.

### 5.3 Les commentaires des avis

In [ ]:
# Avis sans titre et sans message (vide ou composé seulement d'espaces)
sql(f"""
    select
        count(*) as reviews,
        cast(count_if(coalesce(trim(review_comment_title), '') = '') as integer)   as without_title,
        cast(count_if(coalesce(trim(review_comment_message), '') = '') as integer) as without_message,
        cast(count_if(review_comment_message is not null
                      and trim(review_comment_message) = '') as integer)          as only_spaces
    from {csv('olist_order_reviews_dataset')}
""")

**À retenir :**
- Le **titre** manque dans **88 %** des avis : il est presque inutilisable.
- Le **message** manque dans **59 %** des avis, mais il reste **près de 41 000 commentaires** à analyser.
- 9 messages ne contiennent que des espaces : ce ne sont pas de vrais commentaires.

**→ Choix :** le sentiment est calculé sur `review_comment_message`, pas sur le titre.
En staging, un message vide ou fait d'espaces devient `null` (`nullif(trim(...), '')`), pour ne pas fausser le score.

## 6. Les anomalies

**Question :** quelles incohérences trouve-t-on dans les données brutes ? Quatre cas.

### 6.1 Le statut et la date de livraison ne concordent pas toujours

In [ ]:
# Statut « livrée » sans date, ou date de livraison sans le statut « livrée »
sql(f"""
    select
        case
            when order_status = 'delivered' then 'delivered, sans date'
            else 'pas delivered, avec date'
        end      as case_found,
        count(*) as orders
    from {csv('olist_orders_dataset')}
    where (order_status = 'delivered' and order_delivered_customer_date is null)
       or (order_status <> 'delivered' and order_delivered_customer_date is not null)
    group by case_found
""")

**→ Choix :** une commande est considérée comme livrée si elle a une **date de livraison**, quel que soit son statut. La date est plus fiable pour calculer le délai et le retard.

### 6.2 Des noms de colonnes mal orthographiés

In [ ]:
# Colonnes du fichier des produits
sql(f"describe select * from {csv('olist_products_dataset')}")[["column_name", "column_type"]]

**→ Choix :** le fichier des produits n'est pas utilisé dans le pipeline. Pour les fichiers utilisés, le staging corrige les noms trompeurs de la même façon (par exemple `order_delivered_carrier_date`, qui contient une heure, devient `delivered_to_carrier_at`).

### 6.3 Des catégories sans traduction

In [ ]:
# Catégories de produits absentes de la table de traduction
sql(f"""
    select p.product_category_name, count(*) as products
    from {csv('olist_products_dataset')} p
    left join {csv('product_category_name_translation')} t
        on p.product_category_name = t.product_category_name
    where p.product_category_name is not null
      and t.product_category_name is null
    group by p.product_category_name
""")

**→ Choix :** sans dimension produit, la traduction n'est pas utilisée. Si on ajoutait une dimension produit, ces 13 produits garderaient leur nom portugais.

### 6.4 Des paiements étranges

In [ ]:
# Moyen de paiement non défini, ou paiement en 0 versement
sql(f"""
    select payment_type, payment_installments, count(*) as payments
    from {csv('olist_order_payments_dataset')}
    where payment_type = 'not_defined'
       or payment_installments = 0
    group by payment_type, payment_installments
""")

**→ Choix :** les paiements ne sont pas utilisés dans le pipeline (voir la section 8). Ces anomalies n'ont donc pas d'effet sur l'entrepôt.

## 7. La période couverte

**Question :** quelles dates couvrent les données ? Tous les mois sont-ils complets ?

In [ ]:
# Première et dernière date de chaque colonne de date des commandes
sql(f"""
    select
        min(order_purchase_timestamp)::date      as first_purchase,
        max(order_purchase_timestamp)::date      as last_purchase,
        max(order_delivered_customer_date)::date as last_delivery,
        max(order_estimated_delivery_date)::date as last_estimated_delivery
    from {csv('olist_orders_dataset')}
""")

### Le nombre de commandes par mois

In [ ]:
import matplotlib.pyplot as plt

# Commandes par mois d'achat
orders_per_month = sql(f"""
    select
        strftime(order_purchase_timestamp, '%Y-%m') as month,
        count(*)                                    as orders
    from {csv('olist_orders_dataset')}
    group by month
    order by month
""")

orders_per_month.plot.bar(x="month", y="orders", legend=False, figsize=(12, 4),
                          title="Commandes par mois d'achat")
plt.xlabel("")
plt.show()

In [ ]:
# Les mois aux extrémités de la période
orders_per_month[(orders_per_month["month"] < "2017-01") | (orders_per_month["month"] > "2018-08")]

**À retenir :**
- Les achats vont de **septembre 2016 à octobre 2018**, et les livraisons estimées jusqu'en **novembre 2018**.
- Les **premiers mois** (fin 2016) et les **derniers** (septembre et octobre 2018) n'ont presque aucune commande : la plateforme démarre, puis l'extraction s'arrête. Novembre 2016 n'a aucune commande.
- Le pic de **novembre 2017** correspond au **Black Friday**.

**→ Choix :**
- `dim_date` couvre les années **2016 à 2018 en entier**, ce qui inclut toutes les dates des commandes.
- Pour l'analyse, la période complète est **janvier 2017 à août 2018** : les mois aux extrémités fausseraient les tendances.

## 8. Les articles et les paiements

**Question :** pour calculer le chiffre d'affaires, faut-il additionner les **articles** (prix + frais de port) ou les **paiements** ? Donnent-ils le même total ?

In [ ]:
# Total des articles et total des paiements
sql(f"""
    select
        (select round(sum(price + freight_value), 0) from {csv('olist_order_items_dataset')})  as items_total,
        (select round(sum(payment_value), 0)         from {csv('olist_order_payments_dataset')}) as payments_total
""")

Les totaux diffèrent d'environ 165 000 BRL. D'où vient l'écart ?

In [ ]:
# Commandes payées mais sans aucun article, par statut
sql(f"""
    with payments as (
        select order_id, sum(payment_value) as paid
        from {csv('olist_order_payments_dataset')}
        group by order_id
    )
    select
        o.order_status,
        count(*)                    as orders,
        round(sum(payments.paid), 0) as paid
    from payments
    join {csv('olist_orders_dataset')} o on payments.order_id = o.order_id
    where payments.order_id not in (select order_id from {csv('olist_order_items_dataset')})
    group by o.order_status
    order by orders desc
""")

In [ ]:
# Pour les commandes qui ont des articles ET des paiements : les totaux concordent-ils ?
sql(f"""
    with items as (
        select order_id, sum(price + freight_value) as items_amount
        from {csv('olist_order_items_dataset')}
        group by order_id
    ),
    payments as (
        select order_id, sum(payment_value) as paid
        from {csv('olist_order_payments_dataset')}
        group by order_id
    )
    select
        count(*)                                                        as orders,
        cast(count_if(abs(items_amount - paid) < 0.01) as integer)      as same_amount,
        round(sum(items_amount), 0)                                     as items_total,
        round(sum(paid), 0)                                             as payments_total
    from items
    join payments on items.order_id = payments.order_id
""")

**À retenir :**
- L'écart vient surtout de **775 commandes payées mais sans article** : elles sont **indisponibles** ou **annulées** (environ 162 600 BRL).
- Pour les commandes qui ont des articles, les deux totaux sont presque identiques : **99,6 %** des commandes ont exactement le même montant, et l'écart total n'est que d'environ 2 900 BRL.

**→ Choix :** le chiffre d'affaires est calculé à partir des **articles** (prix + frais de port).
On ne compte pas l'argent de commandes jamais honorées. `stg_order_payments` est préparé en staging, mais n'est utilisé par aucun fait.

## 9. Les mots des commentaires

**Question :** quels mots annoncent un bon ou un mauvais avis ? C'est la base du lexique de sentiment.

Méthode : on découpe chaque commentaire en mots (minuscules, sans accents), puis on calcule pour chaque mot fréquent la **note moyenne** des avis qui le contiennent.

In [ ]:
# Note moyenne des avis contenant chaque mot (mots d'au moins 3 lettres, vus au moins 150 fois)
word_scores = sql(f"""
    with words as (
        select
            review_score,
            unnest(regexp_split_to_array(lower(strip_accents(review_comment_message)), '[^a-z]+')) as word
        from {csv('olist_order_reviews_dataset')}
        where trim(review_comment_message) <> ''
    )
    select word, count(*) as occurrences, round(avg(review_score), 2) as avg_score
    from words
    where length(word) >= 3
    group by word
    having count(*) >= 150
    order by avg_score
""")

len(word_scores)

In [ ]:
# Les 15 mots les plus négatifs
word_scores.head(15)

In [ ]:
# Les 15 mots les plus positifs
word_scores.tail(15)

**À retenir :**
- Les extrêmes contiennent des **mots d'opinion** (`pessimo` « très mauvais », `amei` « j'ai adoré »)…
- … mais aussi des mots seulement **liés** aux mauvais avis : `dinheiro` (« argent »), `resposta` (« réponse »). Ce ne sont pas des opinions.

On ne garde donc **que les mots d'opinion**, à la main.

### Les pièges

In [ ]:
# Des mots qui semblent négatifs, mais pas dans leur contexte
word_scores[word_scores["word"].isin(["problemas", "pena", "nao"])]

**À retenir :**
- `problemas` (3,63/5) et `pena` (4,10/5) ont une note **plutôt bonne**, alors qu'ils semblent négatifs : « *sem problemas* » (« sans problème »), « *vale a pena* » (« ça vaut la peine »). On les exclut.
- `nao` (« non / ne… pas ») a une note moyenne d'environ 2/5 : il signale presque toujours un problème (« *não recebi* », « je n'ai pas reçu »). On le garde comme mot **négatif**.

### Le lexique fonctionne-t-il ?

On applique le lexique (`ctc_olist/seeds/sentiment_lexicon.csv`) à tous les commentaires, puis on compare le sentiment obtenu à la note donnée par le client.

In [ ]:
# Score de chaque commentaire = somme des polarités de ses mots présents dans le lexique
validation = sql(f"""
    with words as (
        select
            review_id,
            order_id,
            review_score,
            unnest(regexp_split_to_array(lower(strip_accents(review_comment_message)), '[^a-z]+')) as word
        from {csv('olist_order_reviews_dataset')}
        where trim(review_comment_message) <> ''
    ),
    scores as (
        select words.review_id, words.order_id, any_value(words.review_score) as review_score,
               coalesce(sum(lexicon.polarity), 0) as sentiment_score
        from words
        left join '../ctc_olist/seeds/sentiment_lexicon.csv' lexicon on words.word = lexicon.word
        group by words.review_id, words.order_id
    )
    select
        case when sentiment_score > 0 then 'positive'
             when sentiment_score < 0 then 'negative'
             else 'neutral' end         as sentiment,
        count(*)                        as reviews,
        round(avg(review_score), 2)     as avg_review_score
    from scores
    group by sentiment
    order by avg_review_score
""")

validation

In [ ]:
validation.plot.bar(x="sentiment", y="avg_review_score", legend=False, figsize=(6, 4),
                    title="Note moyenne selon le sentiment calculé", ylim=(0, 5))
plt.xlabel("")
plt.show()

**À retenir :** plus le sentiment calculé est positif, plus la note est élevée. Le lexique, bien que simple, **suit l'avis réel des clients**.

**Limite connue :** la négation n'est pas comprise. « *não é ruim* » (« ce n'est pas mauvais ») compte comme négatif.

**→ Choix :** un lexique de **58 mots d'opinion** (+1 / −1), choisi à partir de ces données, transforme le texte libre (données non structurées, la Variété des 3V) en **score de sentiment** dans `int_review_sentiment`.

> Les chiffres diffèrent un peu de ceux du pipeline : ici, on garde tous les avis, alors que `int_review_sentiment` ne garde que le plus récent de chaque commande (section 4).

## Conclusion

| Section | Observation dans les données brutes | Choix dans le pipeline |
|---|---|---|
| 1 | types devinés, codes postaux avec un 0 en tête | conversions explicites, codes postaux en texte |
| 2 | un `customer_id` par commande | granularité « personne » de `dim_customers` |
| 3 | 252 personnes avec plusieurs adresses | SCD type 1 (écraser l'ancienne valeur) |
| 4 | 547 commandes avec plusieurs avis | avis le plus récent |
| 5 | dates de livraison et commentaires manquants | délai et retard vides, `nullif` sur les commentaires |
| 6 | statuts incohérents, noms trompeurs | livraison définie par la date, noms corrigés en staging |
| 7 | mois incomplets aux extrémités | `dim_date` 2016-2018, analyse 2017-01 à 2018-08 |
| 8 | 775 commandes payées sans article | chiffre d'affaires à partir des articles, paiements non utilisés |
| 9 | des mots annoncent la note | lexique de sentiment |

Chaque choix du pipeline repose sur une observation des données. Le détail est dans `docs/choix_de_modelisation.md`.